In [2]:
import os
os.chdir('..')
print("Working from:", os.getcwd())

Working from: C:\Users\Abhijeet\PhD_MAML_AE


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
import json

np.random.seed(42)
random.seed(42)

TRAIN_PATH = "data/raw/train/"
TEST_PATH  = "data/raw/test/"
LABEL_PATH = "data/raw/labeled_anomalies.csv"

labels_df = pd.read_csv(LABEL_PATH)

smap_channels = labels_df[labels_df['spacecraft']=='SMAP']['chan_id'].tolist()
msl_channels  = labels_df[labels_df['spacecraft']=='MSL']['chan_id'].tolist()
all_channels  = smap_channels + msl_channels

print(f"✅ Loaded — {len(all_channels)} total channels")

✅ Loaded — 82 total channels


In [4]:
def create_windows(data, window_size=30, stride=1):
    """
    Convert a time series into overlapping windows.

    Think of a sliding window like a magnifying glass
    moving along the time series one step at a time.
    Each position of the magnifying glass = one window = one training sample.

    Args:
        data        : numpy array of shape (timesteps, features)
        window_size : how many timesteps per window
        stride      : how many steps to move the window each time

    Returns:
        windows : numpy array of shape (n_windows, window_size, features)
    """
    windows = []
    for start in range(0, len(data) - window_size + 1, stride):
        end = start + window_size
        windows.append(data[start:end])
    return np.array(windows)

# Test it on one channel
sample_data   = np.load(f"{TRAIN_PATH}P-1.npy")
sample_windows = create_windows(sample_data, window_size=30, stride=1)

print("Sliding window demonstration:")
print(f"  Original data shape    : {sample_data.shape}")
print(f"  After windowing        : {sample_windows.shape}")
print(f"  Each window shape      : {sample_windows[0].shape}")
print(f"\nInterpretation:")
print(f"  {sample_windows.shape[0]} windows, each containing")
print(f"  {sample_windows.shape[1]} timesteps × {sample_windows.shape[2]} sensor features")
print(f"\n✅ Sliding window function works correctly")

Sliding window demonstration:
  Original data shape    : (2872, 25)
  After windowing        : (2843, 30, 25)
  Each window shape      : (30, 25)

Interpretation:
  2843 windows, each containing
  30 timesteps × 25 sensor features

✅ Sliding window function works correctly


In [5]:
def get_anomaly_windows(channel_id, window_size=30, stride=5):
    """
    Extract windows that contain anomalous data.
    These become the support set examples in MAML episodes.

    We use stride=5 here (not 1) to avoid too much overlap
    between windows in the support set. This gives more
    diverse examples per shot.
    """
    test_data      = np.load(f"{TEST_PATH}{channel_id}.npy")
    ch_info        = labels_df[labels_df['chan_id'] == channel_id]
    anomaly_seqs   = eval(ch_info['anomaly_sequences'].values[0])

    anomaly_windows = []
    for seq in anomaly_seqs:
        start = int(seq[0])
        end   = int(seq[1])

        # Extract the anomalous segment
        segment = test_data[max(0, start):min(end + window_size, len(test_data))]

        if len(segment) >= window_size:
            wins = create_windows(segment, window_size=window_size, stride=stride)
            anomaly_windows.extend(wins)

    return np.array(anomaly_windows) if anomaly_windows else np.array([])

def get_normal_windows(channel_id, window_size=30, stride=10, max_windows=500):
    """
    Extract windows from normal (training) data.
    These become the query set negative examples in MAML episodes.

    We limit to max_windows to keep memory manageable.
    """
    train_data = np.load(f"{TRAIN_PATH}{channel_id}.npy")
    all_wins   = create_windows(train_data, window_size=window_size, stride=stride)

    # Randomly sample if too many
    if len(all_wins) > max_windows:
        idx      = np.random.choice(len(all_wins), max_windows, replace=False)
        all_wins = all_wins[idx]

    return all_wins

# Test on one channel
ch = 'P-1'
anom_wins   = get_anomaly_windows(ch, window_size=30, stride=5)
normal_wins = get_normal_windows(ch,  window_size=30, stride=10)

print(f"Channel P-1 window extraction:")
print(f"  Anomaly windows : {anom_wins.shape}")
print(f"  Normal windows  : {normal_wins.shape}")
print(f"\nFor MAML episodes:")
print(f"  Support set     — pick K windows from anomaly_windows")
print(f"  Query set       — mix of remaining anomaly + normal windows")

Channel P-1 window extraction:
  Anomaly windows : (152, 30, 25)
  Normal windows  : (285, 30, 25)

For MAML episodes:
  Support set     — pick K windows from anomaly_windows
  Query set       — mix of remaining anomaly + normal windows


In [6]:
print("Processing all channels...")
print("=" * 55)

channel_data = {}
problem_channels = []

for ch in all_channels:
    anom_wins   = get_anomaly_windows(ch, window_size=30, stride=5)
    normal_wins = get_normal_windows(ch,  window_size=30, stride=10)

    if len(anom_wins) == 0:
        problem_channels.append(ch)
        continue

    channel_data[ch] = {
        'anomaly_windows' : anom_wins,
        'normal_windows'  : normal_wins,
        'n_anomaly'       : len(anom_wins),
        'n_normal'        : len(normal_wins),
        'spacecraft'      : 'SMAP' if ch in smap_channels else 'MSL'
    }

# Summary
n_anom  = [v['n_anomaly'] for v in channel_data.values()]
n_norm  = [v['n_normal']  for v in channel_data.values()]

print(f"Channels processed successfully : {len(channel_data)}")
print(f"Channels with no anomaly windows: {len(problem_channels)}")
if problem_channels:
    print(f"  Problem channels: {problem_channels}")

print(f"\nAnomaly windows per channel:")
print(f"  Mean   : {np.mean(n_anom):.1f}")
print(f"  Min    : {np.min(n_anom)}")
print(f"  Max    : {np.max(n_anom)}")

print(f"\nNormal windows per channel:")
print(f"  Mean   : {np.mean(n_norm):.1f}")
print(f"  Min    : {np.min(n_norm)}")
print(f"  Max    : {np.max(n_norm)}")

print(f"\nShot feasibility check:")
for k in [1, 5, 10]:
    suitable = sum(1 for v in channel_data.values() if v['n_anomaly'] >= k)
    print(f"  {k}-shot : {suitable}/{len(channel_data)} channels suitable ✅")

Processing all channels...
Channels processed successfully : 81
Channels with no anomaly windows: 0

Anomaly windows per channel:
  Mean   : 157.3
  Min    : 3
  Max    : 844

Normal windows per channel:
  Mean   : 240.0
  Min    : 29
  Max    : 428

Shot feasibility check:
  1-shot : 81/81 channels suitable ✅
  5-shot : 80/81 channels suitable ✅
  10-shot : 74/81 channels suitable ✅


In [7]:
# Get list of usable channels
usable_channels = list(channel_data.keys())
random.shuffle(usable_channels)

# Split: 60 meta-train, 10 meta-val, 12 meta-test
n_total = len(usable_channels)
n_test  = 12
n_val   = 10
n_train = n_total - n_test - n_val

meta_train_channels = usable_channels[:n_train]
meta_val_channels   = usable_channels[n_train:n_train + n_val]
meta_test_channels  = usable_channels[n_train + n_val:]

print("=" * 55)
print("TASK SPLIT FOR MAML TRAINING")
print("=" * 55)
print(f"Total usable channels : {n_total}")
print(f"Meta-train tasks      : {len(meta_train_channels)}")
print(f"Meta-validation tasks : {len(meta_val_channels)}")
print(f"Meta-test tasks       : {len(meta_test_channels)}")

print(f"\nMeta-test channels (held out — never seen during training):")
print(f"  {meta_test_channels}")

print(f"\nWhy this split matters:")
print(f"  Meta-train  → MAML learns θ* from these 60 tasks")
print(f"  Meta-val    → We tune hyperparameters on these 10 tasks")
print(f"  Meta-test   → Final 1/5/10-shot evaluation on these 12 tasks")
print(f"  The meta-test tasks are NEVER touched during training")

TASK SPLIT FOR MAML TRAINING
Total usable channels : 81
Meta-train tasks      : 59
Meta-validation tasks : 10
Meta-test tasks       : 12

Meta-test channels (held out — never seen during training):
  ['M-7', 'E-3', 'M-6', 'E-10', 'T-13', 'E-12', 'P-2', 'D-6', 'P-4', 'D-8', 'E-2', 'E-13']

Why this split matters:
  Meta-train  → MAML learns θ* from these 60 tasks
  Meta-val    → We tune hyperparameters on these 10 tasks
  Meta-test   → Final 1/5/10-shot evaluation on these 12 tasks
  The meta-test tasks are NEVER touched during training


In [8]:
def build_episode(channel_id, k_shot=5, n_query_anomaly=10,
                  n_query_normal=50):
    """
    Build one MAML episode for a given channel and shot number.

    An episode has two parts:
    - Support set : K anomaly windows (the few examples the model adapts from)
    - Query set   : Mix of anomaly + normal windows (evaluates adaptation)

    Args:
        channel_id       : which channel to build the episode from
        k_shot           : how many anomaly windows in support set (1, 5, or 10)
        n_query_anomaly  : anomaly windows in query set
        n_query_normal   : normal windows in query set

    Returns:
        support_x  : (k_shot, window_size, features)
        query_x    : (n_query, window_size, features)
        query_y    : (n_query,) — 1=anomaly, 0=normal
    """
    data        = channel_data[channel_id]
    anom_wins   = data['anomaly_windows'].copy()
    normal_wins = data['normal_windows'].copy()

    # Shuffle
    np.random.shuffle(anom_wins)
    np.random.shuffle(normal_wins)

    # Support set — first K anomaly windows
    support_x = anom_wins[:k_shot]

    # Query set — remaining anomaly + normal windows
    remaining_anom = anom_wins[k_shot:k_shot + n_query_anomaly]
    query_normal   = normal_wins[:n_query_normal]

    # If not enough remaining anomaly windows, use what we have
    if len(remaining_anom) == 0:
        remaining_anom = anom_wins[:min(3, len(anom_wins))]

    query_x = np.concatenate([remaining_anom, query_normal], axis=0)
    query_y = np.concatenate([
        np.ones(len(remaining_anom)),
        np.zeros(len(query_normal))
    ])

    # Shuffle query set
    idx     = np.random.permutation(len(query_x))
    query_x = query_x[idx]
    query_y = query_y[idx]

    return support_x, query_x, query_y

# Test episode building for all shot conditions
print("Episode building test:")
print("=" * 55)

for k in [1, 5, 10]:
    sx, qx, qy = build_episode('P-1', k_shot=k)
    print(f"\n{k}-shot episode from channel P-1:")
    print(f"  Support set shape  : {sx.shape}")
    print(f"    = {sx.shape[0]} examples × {sx.shape[1]} timesteps × {sx.shape[2]} features")
    print(f"  Query set shape    : {qx.shape}")
    print(f"  Query labels shape : {qy.shape}")
    print(f"  Anomalies in query : {int(qy.sum())} / {len(qy)}")

print(f"\n✅ Episode building works for all shot conditions")

Episode building test:

1-shot episode from channel P-1:
  Support set shape  : (1, 30, 25)
    = 1 examples × 30 timesteps × 25 features
  Query set shape    : (60, 30, 25)
  Query labels shape : (60,)
  Anomalies in query : 10 / 60

5-shot episode from channel P-1:
  Support set shape  : (5, 30, 25)
    = 5 examples × 30 timesteps × 25 features
  Query set shape    : (60, 30, 25)
  Query labels shape : (60,)
  Anomalies in query : 10 / 60

10-shot episode from channel P-1:
  Support set shape  : (10, 30, 25)
    = 10 examples × 30 timesteps × 25 features
  Query set shape    : (60, 30, 25)
  Query labels shape : (60,)
  Anomalies in query : 10 / 60

✅ Episode building works for all shot conditions


In [9]:
import pickle

os.makedirs("data/processed", exist_ok=True)

# Save channel data
with open("data/processed/channel_data.pkl", "wb") as f:
    pickle.dump(channel_data, f)

# Save task splits
task_splits = {
    'meta_train' : meta_train_channels,
    'meta_val'   : meta_val_channels,
    'meta_test'  : meta_test_channels,
}
with open("data/processed/task_splits.json", "w") as f:
    json.dump(task_splits, f, indent=2)

# Save summary
summary = {
    'window_size'        : 30,
    'stride_anomaly'     : 5,
    'stride_normal'      : 10,
    'total_tasks'        : len(channel_data),
    'meta_train_tasks'   : len(meta_train_channels),
    'meta_val_tasks'     : len(meta_val_channels),
    'meta_test_tasks'    : len(meta_test_channels),
    'shot_conditions'    : [1, 5, 10],
}
with open("data/processed/preprocessing_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 55)
print("PREPROCESSING COMPLETE — FILES SAVED")
print("=" * 55)
print(f"\nSaved to data/processed/:")
print(f"  channel_data.pkl          — all windows per channel")
print(f"  task_splits.json          — train/val/test task lists")
print(f"  preprocessing_summary.json — configuration record")

print(f"\nConfiguration locked in:")
for k, v in summary.items():
    print(f"  {k:25s}: {v}")

print(f"\n✅ Notebook 2 — Preprocessing COMPLETE")
print(f"Next: Notebook 3 — Build the LSTM Autoencoder")

PREPROCESSING COMPLETE — FILES SAVED

Saved to data/processed/:
  channel_data.pkl          — all windows per channel
  task_splits.json          — train/val/test task lists
  preprocessing_summary.json — configuration record

Configuration locked in:
  window_size              : 30
  stride_anomaly           : 5
  stride_normal            : 10
  total_tasks              : 81
  meta_train_tasks         : 59
  meta_val_tasks           : 10
  meta_test_tasks          : 12
  shot_conditions          : [1, 5, 10]

✅ Notebook 2 — Preprocessing COMPLETE
Next: Notebook 3 — Build the LSTM Autoencoder


In [10]:
print("Checking normalization across all channels...")
print("=" * 50)

mins, maxs = [], []
problem_channels = []

for ch in all_channels:
    try:
        tr = np.load(f"data/raw/train/{ch}.npy")
        te = np.load(f"data/raw/test/{ch}.npy")
        mins.append(min(tr.min(), te.min()))
        maxs.append(max(tr.max(), te.max()))
    except Exception as e:
        problem_channels.append(ch)

global_min = min(mins)
global_max = max(maxs)

print(f"Global min across all channels : {global_min:.4f}")
print(f"Global max across all channels : {global_max:.4f}")
print(f"Channels checked               : {len(mins)}")

if global_min >= -0.01 and global_max <= 1.01:
    print("\n✅ Data is pre-normalized between 0 and 1")
    print("   No further normalization needed before training")
elif global_min >= -1.01 and global_max <= 1.01:
    print("\n⚠️ Data is between -1 and 1 — standardized not normalized")
    print("   This is still usable but note this in your thesis")
else:
    print(f"\n❌ Data is NOT normalized")
    print(f"   Range: {global_min:.4f} to {global_max:.4f}")
    print(f"   We need to apply MinMaxScaler before training")

Checking normalization across all channels...
Global min across all channels : -1.4772
Global max across all channels : 258.1081
Channels checked               : 82

❌ Data is NOT normalized
   Range: -1.4772 to 258.1081
   We need to apply MinMaxScaler before training


In [11]:
from sklearn.preprocessing import MinMaxScaler
import pickle
import json
import os

print("Applying MinMax Normalization...")
print("=" * 55)
print("Rule: Fit scaler on TRAIN data only.")
print("      Apply same scaler to TEST data.")
print("      This prevents data leakage.\n")

# Window size settings — same as before
WINDOW_SIZE    = 30
STRIDE_ANOMALY = 5
STRIDE_NORMAL  = 10
MAX_NORMAL     = 500

channel_data_normalized = {}
problem_channels        = []

for i, ch in enumerate(all_channels):
    try:
        # Load raw data
        train_raw = np.load(f"data/raw/train/{ch}.npy")
        test_raw  = np.load(f"data/raw/test/{ch}.npy")

        # Fit scaler on training data ONLY
        scaler = MinMaxScaler()
        scaler.fit(train_raw)

        # Apply to both train and test
        train_norm = scaler.transform(train_raw)
        test_norm  = scaler.transform(test_raw)

        # Clip any values slightly outside 0-1 due to test
        # data exceeding training range
        train_norm = np.clip(train_norm, 0, 1)
        test_norm  = np.clip(test_norm,  0, 1)

        # Extract windows from normalized data
        # Anomaly windows from TEST data
        ch_info      = labels_df[labels_df['chan_id'] == ch]
        anomaly_seqs = eval(ch_info['anomaly_sequences'].values[0])

        anomaly_windows = []
        for seq in anomaly_seqs:
            start   = int(seq[0])
            end     = int(seq[1])
            segment = test_norm[max(0, start):min(end + WINDOW_SIZE,
                                                   len(test_norm))]
            if len(segment) >= WINDOW_SIZE:
                wins = create_windows(segment, WINDOW_SIZE, STRIDE_ANOMALY)
                anomaly_windows.extend(wins)

        if len(anomaly_windows) == 0:
            problem_channels.append(ch)
            continue

        anomaly_windows = np.array(anomaly_windows)

        # Normal windows from TRAIN data
        all_normal = create_windows(train_norm, WINDOW_SIZE, STRIDE_NORMAL)
        if len(all_normal) > MAX_NORMAL:
            idx        = np.random.choice(len(all_normal),
                                          MAX_NORMAL, replace=False)
            all_normal = all_normal[idx]

        channel_data_normalized[ch] = {
            'anomaly_windows' : anomaly_windows,
            'normal_windows'  : all_normal,
            'n_anomaly'       : len(anomaly_windows),
            'n_normal'        : len(all_normal),
            'spacecraft'      : 'SMAP' if ch in smap_channels else 'MSL',
            'scaler_min'      : scaler.data_min_.tolist(),
            'scaler_max'      : scaler.data_max_.tolist(),
        }

    except Exception as e:
        problem_channels.append(ch)
        print(f"⚠️ Problem with {ch}: {e}")

    if (i+1) % 20 == 0:
        print(f"Progress: {i+1}/{len(all_channels)} channels done")

print(f"\nChannels normalized successfully : {len(channel_data_normalized)}")
print(f"Problem channels                 : {len(problem_channels)}")
if problem_channels:
    print(f"  {problem_channels}")

Applying MinMax Normalization...
Rule: Fit scaler on TRAIN data only.
      Apply same scaler to TEST data.
      This prevents data leakage.

Progress: 20/82 channels done
Progress: 40/82 channels done
Progress: 60/82 channels done
Progress: 80/82 channels done

Channels normalized successfully : 81
Problem channels                 : 0


In [12]:
print("Verifying normalization...")
print("=" * 55)

all_mins, all_maxs = [], []

for ch, data in channel_data_normalized.items():
    all_mins.append(data['anomaly_windows'].min())
    all_mins.append(data['normal_windows'].min())
    all_maxs.append(data['anomaly_windows'].max())
    all_maxs.append(data['normal_windows'].max())

global_min = min(all_mins)
global_max = max(all_maxs)

print(f"Global min after normalization : {global_min:.4f}")
print(f"Global max after normalization : {global_max:.4f}")

if global_min >= -0.01 and global_max <= 1.01:
    print("\n✅ Normalization successful — all values between 0 and 1")
else:
    print(f"\n⚠️ Some values outside 0-1 range")
    print(f"This is usually minor clipping from test data exceeding")
    print(f"training range. The np.clip() handles this automatically.")

# Check shot feasibility still holds
print("\nShot feasibility after normalization:")
for k in [1, 5, 10]:
    suitable = sum(1 for v in channel_data_normalized.values()
                   if v['n_anomaly'] >= k)
    print(f"  {k}-shot : {suitable}/{len(channel_data_normalized)} "
          f"channels suitable ✅")

Verifying normalization...
Global min after normalization : 0.0000
Global max after normalization : 1.0000

✅ Normalization successful — all values between 0 and 1

Shot feasibility after normalization:
  1-shot : 81/81 channels suitable ✅
  5-shot : 80/81 channels suitable ✅
  10-shot : 74/81 channels suitable ✅


In [13]:
# Update channel list to normalized channels only
usable_channels_norm = list(channel_data_normalized.keys())
random.shuffle(usable_channels_norm)

n_total = len(usable_channels_norm)
n_test  = 12
n_val   = 10
n_train = n_total - n_test - n_val

meta_train_channels = usable_channels_norm[:n_train]
meta_val_channels   = usable_channels_norm[n_train:n_train + n_val]
meta_test_channels  = usable_channels_norm[n_train + n_val:]

print("Task split after normalization:")
print(f"  Meta-train : {len(meta_train_channels)} tasks")
print(f"  Meta-val   : {len(meta_val_channels)} tasks")
print(f"  Meta-test  : {len(meta_test_channels)} tasks")

# Save normalized channel data
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/channel_data_normalized.pkl", "wb") as f:
    pickle.dump(channel_data_normalized, f)
print("\n✅ Saved: data/processed/channel_data_normalized.pkl")

# Save task splits
task_splits = {
    'meta_train' : meta_train_channels,
    'meta_val'   : meta_val_channels,
    'meta_test'  : meta_test_channels,
}
with open("data/processed/task_splits.json", "w") as f:
    json.dump(task_splits, f, indent=2)
print("✅ Saved: data/processed/task_splits.json")

# Save preprocessing config
config = {
    'window_size'        : WINDOW_SIZE,
    'stride_anomaly'     : STRIDE_ANOMALY,
    'stride_normal'      : STRIDE_NORMAL,
    'max_normal_windows' : MAX_NORMAL,
    'normalization'      : 'MinMaxScaler per channel — fit on train only',
    'total_tasks'        : len(channel_data_normalized),
    'meta_train_tasks'   : len(meta_train_channels),
    'meta_val_tasks'     : len(meta_val_channels),
    'meta_test_tasks'    : len(meta_test_channels),
    'shot_conditions'    : [1, 5, 10],
    'query_anomaly'      : 10,
    'query_normal'       : 50,
}
with open("data/processed/config.json", "w") as f:
    json.dump(config, f, indent=2)
print("✅ Saved: data/processed/config.json")

print("\n" + "=" * 55)
print("PREPROCESSING FULLY COMPLETE")
print("=" * 55)
print("\nAll data is now:")
print("  ✅ Normalized between 0 and 1")
print("  ✅ Windowed into (30, 25) samples")
print("  ✅ Split into support and query ready episodes")
print("  ✅ Task splits saved for reproducibility")
print("\nNext: Notebook 3 — Build the LSTM Autoencoder")

Task split after normalization:
  Meta-train : 59 tasks
  Meta-val   : 10 tasks
  Meta-test  : 12 tasks

✅ Saved: data/processed/channel_data_normalized.pkl
✅ Saved: data/processed/task_splits.json
✅ Saved: data/processed/config.json

PREPROCESSING FULLY COMPLETE

All data is now:
  ✅ Normalized between 0 and 1
  ✅ Windowed into (30, 25) samples
  ✅ Split into support and query ready episodes
  ✅ Task splits saved for reproducibility

Next: Notebook 3 — Build the LSTM Autoencoder
